# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is fully FAIR and AI-ready.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect high-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show summary metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview
Enumerate all record sets and the fields (columns) defined, referencing each entity's `@id`.

We'll print all available record sets, their field details (names and `@id`s), and sample records.

In [ ]:
# List all record sets and their fields (using `@id`s)

print("Available record sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - {rs['@id']} : {rs['name']} ({rs.get('description', '')})")

print("\nFor each record set, list fields (columns):")
for rs in record_sets:
    print(f"\nRecord Set {rs['@id']} fields:")
    for field in rs.get('field', []):
        # Each field is a dict with '@id', 'name', 'description', 'dataType', etc.
        print(f"    - {field['@id']}: {field.get('name', '')} [{field.get('dataType', '')}] -- {field.get('description', '')}")
    print("Sample record:")
    for i, record in enumerate(dataset.records(record_set=rs['@id'])):
        print(record)
        if i == 0:
            break


## 3. Data Extraction
Extract all tabular data into pandas DataFrames from each record set. We'll use the `@id` of the record set, and you can choose which set and fields to analyze further.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for {record_set_id}")

# Example: show columns and head for the main survey record set
# (choose main tabular data set by inspecting previous cell's output)
main_record_set_id = record_set_ids[0]  # Update if your main data set is different
print(f"\nColumns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate filtering, normalization, and grouping using numeric and categorical fields.

_**Customize** the field references as appropriate for your analysis!_

In [ ]:
# Choose record set and numeric field (reference by @id):
record_set_id = main_record_set_id  # Could be e.g. 'cr:SurveyResponses' if that's in your metadata
df = dataframes[record_set_id]

# Choose an example numeric field. Replace with a known field '@id' (use previous output)
numeric_field_id = 'cr:GAD7_total'  # Example field id for GAD-7 total score
# If this field does not exist, replace with the appropriate field id ('cr:PHQ9_total', etc.)

# Filter records where the selected numeric score is above a threshold
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Example grouping by a categorical field (such as education or gender)
    group_field_id = 'cr:level_of_education'  # Use an appropriate field id from metadata
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of the selected numeric field and compare distributions by groups, if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id in df.columns:
    df[numeric_field_id].hist(bins=15, edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group
    if group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We've successfully loaded and explored the FAIR² Mental Health Survey dataset using `mlcroissant`. This notebook demonstrated:
- Loading Croissant metadata and enumerating record sets and field IDs.
- Extracting data into pandas DataFrames for analysis.
- Performing basic exploratory data analysis (filtering, normalization, grouping).
- Visualizing selected data distributions.

The dataset enables further research on mental health and demographic patterns in Kilifi County, Kenya, and demonstrates AI-ready best practices for FAIR social/health surveys.

_Feel free to extend this notebook by adding deeper modeling, more visualizations, or linking additional FAIR datasets!_